# Nordeus Job Fair 2026 — Clan Tournament Winner Prediction

**Author:** Marija Gijic  
**Task:** Binary classification — predict which clan wins a tournament match (`clan_winner` = 1 or 2)  
**Metric:** Accuracy  
**Final CV accuracy:** 0.5836 ± 0.0074 (LightGBM + XGBRegressor on score differential, blended)

## Approach
1. **EDA** — identify what separates winners from losers at player and clan level  
2. **Feature engineering** — 89 clan-level features including per-rank matchup features that model the best-vs-best pairing mechanic  
3. **Modeling** — LightGBM classifier blended with XGBRegressor trained on point differential (not binary win/loss) — the second model treats a 60–0 win as stronger evidence than a 10–9 win  
4. **BONUS** — Smart advisory chatbot: tells clan leaders exactly which members to focus on and why

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, json
warnings.filterwarnings('ignore')

import xgboost as xgb
import lightgbm as lgb
from xgboost import XGBRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.3f}'.format)

DATA = '/content/drive/MyDrive/Job_Fair 2026__Data_Science_Challenge/'
# DATA = 'data/'  # local

members_train = pd.read_csv(DATA + 'member_stats_training.csv')
matches_train = pd.read_csv(DATA + 'clan_matches_training.csv')
members_test  = pd.read_csv(DATA + 'member_stats_test.csv')
matches_test  = pd.read_csv(DATA + 'clan_matches_test.csv')

print(f'members_train : {members_train.shape}')
print(f'matches_train : {matches_train.shape}')
print(f'members_test  : {members_test.shape}')
print(f'matches_test  : {matches_test.shape}')
print(f'\nTarget balance:\n{matches_train["clan_winner"].value_counts(normalize=True).round(4)}')

---
## Part 1 — Exploratory Data Analysis

### 1.1 What separates winners from losers at player level?

In [ ]:
# Tag each clan appearance as winner or loser
clan_results = {}
for _, row in matches_train.iterrows():
    clan_results[row['clan_1_id']] = 'winner' if row['clan_winner'] == 1 else 'loser'
    clan_results[row['clan_2_id']] = 'winner' if row['clan_winner'] == 2 else 'loser'
members_train['result'] = members_train['clan_id'].map(clan_results)

num_cols = [
    'days_active_last_28_days', 'days_active_last_7_days', 'days_since_last_active',
    'training_count_last_28_days', 'clan_multiplier',
    'avg_stars_top_11_players', 'avg_stars_top_3_players', 'avg_training_bonus', 'cohort_day'
]
rows = []
for col in num_cols:
    w = members_train[members_train['result'] == 'winner'][col].mean()
    l = members_train[members_train['result'] == 'loser'][col].mean()
    rows.append({'feature': col, 'winner_mean': w, 'loser_mean': l, 'diff_%': (w - l) / abs(l) * 100})
pd.DataFrame(rows).sort_values('diff_%', key=abs, ascending=False).round(3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
focus = [
    ('training_count_last_28_days', 'Training sessions (28d)'),
    ('avg_training_bonus',           'Avg training bonus'),
    ('days_since_last_active',        'Days since last login'),
    ('avg_stars_top_11_players',      'Avg stars top-11'),
    ('clan_multiplier',               'Clan multiplier'),
    ('days_active_last_7_days',       'Days active last 7d'),
]
for ax, (col, label) in zip(axes.flat, focus):
    for res, color in [('winner', '#2ecc71'), ('loser', '#e74c3c')]:
        ax.hist(members_train[members_train['result'] == res][col].dropna(),
                bins=40, alpha=0.6, label=res, color=color, density=True)
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('Player-level Distributions: Winners vs Losers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.2 The Weakest-Link Effect

Because every member's points are multiplied by their `clan_multiplier`, **one inactive player drags the whole team**. A ghost member (>14 days inactive) effectively contributes 0 points regardless of their quality.

In [ ]:
pen = members_train.groupby('clan_id').apply(lambda d: pd.Series({
    'ghost_count':    (d['days_since_last_active'] > 14).sum(),
    'full_attendance': int(d['days_since_last_active'].max() == 0),
    'result':          d['result'].iloc[0]
})).reset_index()

bonus_agg = members_train.groupby('clan_id').agg(
    min_bonus=('avg_training_bonus', 'min'),
    result=('result', 'first')
).reset_index()
bonus_agg['min_bonus_tier'] = pd.cut(
    bonus_agg['min_bonus'], bins=[-0.1, 0.05, 3, 8, 12, 18.5],
    labels=['0 (none)', '0–3', '3–8', '8–12', '12+']
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ghost_wr = pen.groupby('ghost_count').apply(lambda d: (d['result'] == 'winner').mean()).reset_index()
ghost_wr.columns = ['ghost_count', 'win_rate']
axes[0].bar(ghost_wr['ghost_count'], ghost_wr['win_rate'], color='#e74c3c', edgecolor='white')
axes[0].axhline(0.5, color='gray', linestyle='--')
axes[0].set_xlabel('Ghost members (>14d inactive)')
axes[0].set_ylabel('Win rate')
axes[0].set_title('Ghost Members → Win Rate')

wr_bonus = bonus_agg.groupby('min_bonus_tier', observed=True).apply(
    lambda d: (d['result'] == 'winner').mean()).reset_index()
wr_bonus.columns = ['tier', 'win_rate']
axes[1].bar(range(len(wr_bonus)), wr_bonus['win_rate'], color='#3498db', edgecolor='white')
axes[1].axhline(0.5, color='gray', linestyle='--')
axes[1].set_xticks(range(len(wr_bonus)))
axes[1].set_xticklabels(wr_bonus['tier'].astype(str), rotation=20)
axes[1].set_title('Min Training Bonus → Win Rate')

att_wr = pen.groupby('full_attendance').apply(
    lambda d: (d['result'] == 'winner').mean()).reset_index()
att_wr.columns = ['full_attendance', 'win_rate']
axes[2].bar(['Not full', 'Full attendance'], att_wr['win_rate'],
            color=['#e67e22', '#27ae60'], edgecolor='white')
axes[2].axhline(0.5, color='gray', linestyle='--')
axes[2].set_title('Full Attendance Today → Win Rate')

plt.suptitle('Key Win/Loss Factors', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key numbers:')
print(f'  0 ghost members → {pen[pen.ghost_count==0]["result"].eq("winner").mean():.1%} win rate')
print(f'  4 ghost members → {pen[pen.ghost_count==4]["result"].eq("winner").mean():.1%} win rate')
print(f'  Full attendance → {pen[pen.full_attendance==1]["result"].eq("winner").mean():.1%} win rate')
print(f'  No attendance   → {pen[pen.full_attendance==0]["result"].eq("winner").mean():.1%} win rate')

---
## Part 2 — Feature Engineering

**Strategy:** Aggregate member stats to clan level, then compute pairwise differences (clan_1 − clan_2). The model learns relative advantage, not absolute values.

**Key insight from game mechanics:** Matches are rank-sorted — best player vs best player, 2nd vs 2nd, etc. So we sort each clan's 6 members by quality and create **per-position features**. `diff_r3_expected_pts_max` tells the model exactly who has the scoring advantage at position 3.

| Feature group | Count | What it captures |
|---|---|---|
| Base aggregations | 41 | mean/min/max/std of activity, bonus, quality, multiplier |
| Per-rank matchup | 48 | 6 positions × 8 stats — direct matchup signal |
| **Total clan features** | **89** | per clan |
| Diff features | 89 | clan_1 − clan_2 for all above |
| Ratio features | 8 | clan_1 / clan_2 for key metrics |
| Derived matchup | 2 | n_rank_advantages, sum_rank_pts_edge |
| **Total match features** | **99** | per match |

In [ ]:
def build_per_rank_features(members_df: pd.DataFrame) -> pd.DataFrame:
    df = members_df.copy()
    df['weighted_quality'] = df['clan_multiplier'] * df['avg_stars_top_11_players']
    df['expected_pts_max'] = df['clan_multiplier'] * 3.0
    df['training_eff']     = df['training_count_last_28_days'] / (df['days_active_last_28_days'] + 0.01)
    df['qrank'] = (
        df.groupby('clan_id')['avg_stars_top_11_players']
          .rank(ascending=False, method='first').astype(int).clip(1, 6)
    )
    RANK_COLS = ['avg_stars_top_11_players', 'clan_multiplier', 'avg_training_bonus',
                 'days_active_last_7_days', 'days_since_last_active',
                 'weighted_quality', 'expected_pts_max', 'training_eff']
    parts = []
    for r in range(1, 7):
        sub = df[df['qrank'] == r].set_index('clan_id')[RANK_COLS].copy()
        sub.columns = [f'r{r}_{c}' for c in RANK_COLS]
        parts.append(sub)
    return pd.concat(parts, axis=1)


def build_clan_features(members_df: pd.DataFrame) -> pd.DataFrame:
    df = members_df.copy()
    df['weighted_quality']    = df['clan_multiplier'] * df['avg_stars_top_11_players']
    df['expected_score']      = df['clan_multiplier'] * 3.0
    df['recency_ratio']       = df['days_active_last_7_days'] / (df['days_active_last_28_days'] / 4 + 0.01)
    df['training_efficiency'] = df['training_count_last_28_days'] / (df['days_active_last_28_days'] + 0.01)
    df['is_inactive']         = (df['days_since_last_active'] > 7).astype(int)
    df['is_ghost']            = (df['days_since_last_active'] > 14).astype(int)
    g = df.groupby('clan_id')
    base = pd.DataFrame({
        'mean_days_active_28':      g['days_active_last_28_days'].mean(),
        'min_days_active_28':       g['days_active_last_28_days'].min(),
        'mean_days_active_7':       g['days_active_last_7_days'].mean(),
        'min_days_active_7':        g['days_active_last_7_days'].min(),
        'max_days_since_active':    g['days_since_last_active'].max(),
        'mean_days_since_active':   g['days_since_last_active'].mean(),
        'inactive_count':           g['is_inactive'].sum(),
        'ghost_count':              g['is_ghost'].sum(),
        'all_active_7':             g['days_active_last_7_days'].min().gt(0).astype(int),
        'full_attendance':          g['days_since_last_active'].max().eq(0).astype(int),
        'mean_recency_ratio':       g['recency_ratio'].mean(),
        'min_recency_ratio':        g['recency_ratio'].min(),
        'mean_training_count':      g['training_count_last_28_days'].mean(),
        'min_training_count':       g['training_count_last_28_days'].min(),
        'std_training_count':       g['training_count_last_28_days'].std(),
        'training_efficiency':      g['training_efficiency'].mean(),
        'min_training_efficiency':  g['training_efficiency'].min(),
        'mean_training_bonus':      g['avg_training_bonus'].mean(),
        'min_training_bonus':       g['avg_training_bonus'].min(),
        'max_training_bonus':       g['avg_training_bonus'].max(),
        'std_training_bonus':       g['avg_training_bonus'].std(),
        'bonus_cv':                 g['avg_training_bonus'].std() / (g['avg_training_bonus'].mean() + 0.01),
        'mean_stars_top11':         g['avg_stars_top_11_players'].mean(),
        'min_stars_top11':          g['avg_stars_top_11_players'].min(),
        'max_stars_top11':          g['avg_stars_top_11_players'].max(),
        'std_stars_top11':          g['avg_stars_top_11_players'].std(),
        'mean_stars_top3':          g['avg_stars_top_3_players'].mean(),
        'mean_multiplier':          g['clan_multiplier'].mean(),
        'max_multiplier':           g['clan_multiplier'].max(),
        'sum_multiplier':           g['clan_multiplier'].sum(),
        'sum_expected_score':       g['expected_score'].sum(),
        'sum_weighted_quality':     g['weighted_quality'].sum(),
        'mean_weighted_quality':    g['weighted_quality'].mean(),
        'max_weighted_quality':     g['weighted_quality'].max(),
        'payer_ratio':              g['is_payer_lifetime'].apply(lambda x: (x == True).mean()),
        'whale_count':              g['dynamic_payment_segment'].apply(lambda x: (x == '4) Whale').sum()),
        'mean_cohort_day':          g['cohort_day'].mean(),
        'bonus_x_activity':         g['avg_training_bonus'].mean() * g['days_active_last_7_days'].mean(),
        'quality_x_activity':       g['avg_stars_top_11_players'].mean() * g['days_active_last_7_days'].mean(),
        'min_bonus_x_min_activity': g['avg_training_bonus'].min() * g['days_active_last_7_days'].min(),
        'exp_score_x_activity':     g['expected_score'].sum() * g['days_active_last_7_days'].mean(),
    })
    return pd.concat([base, build_per_rank_features(members_df)], axis=1)


KEY_RATIO_COLS = ['mean_stars_top11', 'sum_expected_score', 'mean_training_bonus',
                  'min_training_bonus', 'sum_multiplier', 'mean_training_count',
                  'sum_weighted_quality', 'mean_days_active_7']

def make_match_features(matches_df: pd.DataFrame, clan_agg: pd.DataFrame) -> pd.DataFrame:
    FEAT_COLS = clan_agg.columns.tolist()
    c1 = clan_agg.reindex(matches_df['clan_1_id'].values)
    c2 = clan_agg.reindex(matches_df['clan_2_id'].values)
    c1.index = matches_df.index
    c2.index = matches_df.index
    diff = c1.subtract(c2)
    diff.columns = [f'diff_{col}' for col in FEAT_COLS]
    for col in KEY_RATIO_COLS:
        diff[f'ratio_{col}'] = c1[col].values / (c2[col].abs().values + 0.1)
    rank_pts_cols = [f'diff_r{r}_expected_pts_max' for r in range(1, 7)]
    diff['n_rank_advantages'] = (diff[rank_pts_cols] > 0).sum(axis=1).astype(float)
    diff['sum_rank_pts_edge'] = diff[rank_pts_cols].sum(axis=1)
    return diff.reset_index(drop=True)


print('Building features...')
clan_agg_train = build_clan_features(members_train)
clan_agg_test  = build_clan_features(members_test)

X_train = make_match_features(matches_train, clan_agg_train)
y_train = (matches_train['clan_winner'] == 1).astype(int).values
X_test  = make_match_features(matches_test, clan_agg_test)
y_diff  = (matches_train['clan_1_points'] - matches_train['clan_2_points']).values.astype(float)

print(f'Clan features  : {clan_agg_train.shape[1]}')
print(f'Match features : {X_train.shape}  (train)')
print(f'Match features : {X_test.shape}   (test)')

---
## Part 3 — Model Training & Cross-Validation

**Final model:** LightGBM classifier + XGBRegressor on point differential, averaged.

**Why two models instead of one?**

The LightGBM classifier learns from binary labels (win=1 / loss=0).  
The XGBRegressor learns from `clan_1_points − clan_2_points` — a continuous score.

The key difference: a 60–0 blowout and a 10–9 narrow win are both "win=1" to the classifier. The regressor sees they are very different — one is strong evidence, the other is noise. It is not "ordinal regression" in the statistical sense (which models ordered categories); it is standard regression used as a richer target signal, then converted back to a binary prediction via `sign(predicted_gap)`.

The two models make different errors — the regressor is stronger on close/uncertain matches, the classifier on clear mismatches — so their simple average outperforms either alone.

**Symmetry augmentation:** for each match, add its mirror image (swap clan_1 ↔ clan_2, negate label and score). Doubles training data and enforces antisymmetry — predicting clan_1 wins must equal predicting clan_2 loses.

In [ ]:
LGB_PARAMS = dict(
    n_estimators=800, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.40, min_child_samples=20,
    reg_alpha=0.1, reg_lambda=2.0, random_state=42, n_jobs=-1, verbose=-1
)
ORD_PARAMS = dict(
    n_estimators=800, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.40, min_child_weight=5,
    gamma=1.0, reg_alpha=0.1, reg_lambda=2.0,
    random_state=42, n_jobs=-1, verbosity=0
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X_train))
oof_ord = np.zeros(len(X_train))
test_lgb = np.zeros(len(X_test))
test_ord = np.zeros(len(X_test))

lgb_scores, ord_scores, blend_scores = [], [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr_raw, y_tr_raw = X_train.iloc[tr_idx], y_train[tr_idx]
    y_diff_raw = y_diff[tr_idx]

    # Symmetry augmentation (training fold only)
    X_tr  = pd.concat([X_tr_raw, -X_tr_raw], ignore_index=True)
    y_tr  = np.concatenate([y_tr_raw, 1 - y_tr_raw])
    y_tr_diff = np.concatenate([y_diff_raw, -y_diff_raw])

    X_val, y_val = X_train.iloc[val_idx], y_train[val_idx]

    # LightGBM classifier
    lgb_m = lgb.LGBMClassifier(**LGB_PARAMS)
    lgb_m.fit(X_tr, y_tr)
    p_lgb = lgb_m.predict_proba(X_val)[:, 1]
    oof_lgb[val_idx] = p_lgb
    test_lgb += lgb_m.predict_proba(X_test)[:, 1] / 5

    # XGBoost ordinal regressor
    ord_m = XGBRegressor(**ORD_PARAMS)
    ord_m.fit(X_tr, y_tr_diff)
    pred_gap = ord_m.predict(X_val)
    gap_std = pred_gap.std() + 1e-6
    p_ord = 1 / (1 + np.exp(-pred_gap / gap_std))
    oof_ord[val_idx] = p_ord
    test_gap = ord_m.predict(X_test)
    test_ord += (1 / (1 + np.exp(-test_gap / gap_std))) / 5

    # Blend
    p_blend = (p_lgb + p_ord) / 2

    s_lgb   = accuracy_score(y_val, (p_lgb   >= 0.5).astype(int))
    s_ord   = accuracy_score(y_val, (p_ord   >= 0.5).astype(int))
    s_blend = accuracy_score(y_val, (p_blend >= 0.5).astype(int))

    lgb_scores.append(s_lgb)
    ord_scores.append(s_ord)
    blend_scores.append(s_blend)

    print(f'Fold {fold+1}: LGB={s_lgb:.4f}  Ordinal={s_ord:.4f}  Blend={s_blend:.4f}')

print(f'\n{"Model":<18} {"Mean":>8} {"Std":>8}')
print('-' * 36)
for name, scores in [("LGB", lgb_scores), ("Ordinal", ord_scores), ("LGB+Ordinal", blend_scores)]:
    print(f'{name:<18} {np.mean(scores):>8.4f} {np.std(scores):>8.4f}')
print(f'{"v1 baseline":<18} {0.5851:>8.4f} {0.0136:>8.4f}')

In [ ]:
# Baseline fold scores from 01_baseline_prototyping.ipynb (XGB+LGB simple average).
# Folds are IDENTICAL: StratifiedKFold uses only len(X) and y for splitting,
# so same n_splits=5, shuffle=True, random_state=42 and same y_train → same validation indices.
V1_FOLD_SCORES = [0.5815, 0.5624, 0.6042, 0.5899, 0.5874]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(1, 6)
axes[0].bar(x - 0.2, V1_FOLD_SCORES, 0.35, label='Baseline (XGB+LGB avg)', color='#95a5a6')
axes[0].bar(x + 0.2, blend_scores,   0.35, label='Final (LGB + Score reg)', color='#3498db')
axes[0].axhline(np.mean(V1_FOLD_SCORES), color='gray',    linestyle='--', linewidth=1,
                label=f'Baseline mean {np.mean(V1_FOLD_SCORES):.4f}')
axes[0].axhline(np.mean(blend_scores),   color='#3498db', linestyle='--', linewidth=1,
                label=f'Final mean {np.mean(blend_scores):.4f}')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Fold {i}' for i in range(1, 6)])
axes[0].set_ylabel('CV Accuracy')
axes[0].set_title('Per-fold comparison\n(same validation sets — comparable)')
axes[0].legend(fontsize=8)
axes[0].set_ylim(0.54, 0.63)

models = ['Baseline\n(XGB+LGB)', 'LGB\nclassifier', 'XGBRegressor\n(score diff)', 'LGB + Score reg\n(final)']
means  = [np.mean(V1_FOLD_SCORES), np.mean(lgb_scores), np.mean(ord_scores), np.mean(blend_scores)]
stds   = [np.std(V1_FOLD_SCORES),  np.std(lgb_scores),  np.std(ord_scores),  np.std(blend_scores)]
colors = ['#95a5a6', '#f39c12', '#9b59b6', '#27ae60']
bars   = axes[1].bar(models, means, color=colors, edgecolor='white')
axes[1].errorbar(range(len(models)), means, yerr=stds,
                 fmt='none', color='black', capsize=5, linewidth=2)
axes[1].set_ylabel('Mean CV Accuracy ± std')
axes[1].set_title('Model comparison (error bars = ±1 std)\nLower std = more consistent on unseen data')
axes[1].set_ylim(0.565, 0.605)
for bar, m, s in zip(bars, means, stds):
    axes[1].text(bar.get_x() + bar.get_width()/2, m + s + 0.001,
                 f'{m:.4f}', ha='center', fontsize=8)

plt.suptitle('Baseline vs Final Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 4 — Feature Importance & Key Findings

In [ ]:
# Train LGB on full augmented data for importance
X_aug = pd.concat([X_train, -X_train], ignore_index=True)
y_aug = np.concatenate([y_train, 1 - y_train])

lgb_full = lgb.LGBMClassifier(**LGB_PARAMS)
lgb_full.fit(X_aug, y_aug)

imp = pd.Series(lgb_full.feature_importances_, index=X_train.columns)
imp = imp / imp.sum()
imp = imp.sort_values(ascending=False)

GROUP_KEYS = [
    ('per_rank',   '#8e44ad', lambda f: f.startswith('diff_r')),
    ('ratio',      '#1abc9c', lambda f: f.startswith('ratio_')),
    ('bonus',      '#e74c3c', lambda f: 'bonus' in f and not f.startswith('diff_r') and not f.startswith('ratio_')),
    ('activity',   '#2ecc71', lambda f: any(k in f for k in ['active','ghost','attend','inactive','recency']) and not f.startswith('diff_r')),
    ('quality',    '#3498db', lambda f: 'star' in f and not f.startswith('diff_r')),
    ('training',   '#f39c12', lambda f: 'train' in f and 'bonus' not in f and not f.startswith('diff_r')),
    ('multiplier', '#9b59b6', lambda f: any(k in f for k in ['multiplier','expected','weighted']) and not f.startswith('diff_r')),
    ('matchup',    '#16a085', lambda f: 'n_rank' in f or 'sum_rank' in f),
]
group_sums = {name: imp[[f for f in imp.index if cond(f)]].sum() for name, _, cond in GROUP_KEYS}
group_sums['other'] = 1 - sum(group_sums.values())

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

top25 = imp.head(25)
colors = []
for f in top25.index:
    c = '#95a5a6'
    for _, col, cond in GROUP_KEYS:
        if cond(f): c = col; break
    colors.append(c)
axes[0].barh(top25.index[::-1], top25.values[::-1], color=colors[::-1])
axes[0].set_xlabel('Normalised importance (LGB)')
axes[0].set_title('Top 25 Features')

pie_data = [(k, v) for k, v in group_sums.items() if v > 0.005]
pie_cols = [next(c for n, c, _ in GROUP_KEYS if n == k) if k != 'other' else '#95a5a6'
            for k, _ in pie_data]
axes[1].pie([v for _, v in pie_data],
            labels=[f'{k}\n({v*100:.1f}%)' for k, v in pie_data],
            colors=pie_cols, startangle=90)
axes[1].set_title('Feature Group Importance')

plt.suptitle('Feature Importance (LGB on full augmented data)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 10 individual features:')
print(imp.head(10).to_string())

In [ ]:
print("""
=== KEY FINDINGS FROM DATA ===

Rank  Factor                          Evidence
────  ──────────────────────────────  ──────────────────────────────────────────────
 1    Min training bonus (floor)       0 bonus → 45% WR │ 12+ bonus → 59% WR (+14pp)
 2    Full attendance (all 6 today)    56% vs 47% win rate — 9pp gap
 3    Ghost members (>14d inactive)    0 ghosts → 53% │ 4 ghosts → 39% WR
 4    Bonus uniformity (low spread)    Uniform teams 56% │ Chaotic teams 46% WR
 5    Training efficiency              Winners train 19% more per active day
 6    Per-rank matchup advantage       Position-3 pts edge reveals hidden advantages
 7    Squad quality (stars)            Matters, but less than engagement metrics
 8    Multiplier carry strategy        max_multiplier is the LEAST important feature

Core insight: FLOOR beats CEILING.
One ghost member destroys a team's chances regardless of how strong the other 5 are.
The weakest-link effect is amplified by the multiplier mechanic — inactive players
contribute 0 pts multiplied by their (potentially high) multiplier.
""")

---
## Part 5 — Final Predictions

In [ ]:
# Final blend: average of 5-fold accumulated LGB + Ordinal probabilities
p_final = (test_lgb + test_ord) / 2
predicted_clan_winner = np.where(p_final >= 0.5, 1, 2)

submission = pd.DataFrame({
    'clan_1_id': matches_test['clan_1_id'],
    'clan_2_id': matches_test['clan_2_id'],
    'predicted_clan_winner': predicted_clan_winner
})
submission.to_csv('clan_winner_predictions.csv', index=False)

print(f'Predictions saved: {len(submission)} rows')
print(f'CV accuracy: {np.mean(blend_scores):.4f} ± {np.std(blend_scores):.4f}')
print(f'\nPrediction distribution:')
print(submission['predicted_clan_winner'].value_counts())
print(f'\nBalance: {(predicted_clan_winner == 1).mean():.3f} clan_1 / {(predicted_clan_winner == 2).mean():.3f} clan_2')
submission.head(10)

---
## Part 6 — BONUS: Smart Clan Advisory Chatbot

The model tells us *who* wins — the advisory system tells the clan leader *what to do about it*.

**How it works:**
1. Load a clan's real player stats from the dataset
2. Compare each metric against benchmarks derived from the EDA (e.g. min_bonus should be ≥ 8 for competitive clans)
3. Identify the specific players who are the weakest links
4. Produce a prioritised action plan
5. **(With Claude API)** Answer free-text questions conversationally

**Business value:** Targeted advice based on real match data → higher clan engagement → better retention → more organic revenue. No dark patterns — recommendations are grounded in actual win-rate data.

In [ ]:
# ── Benchmarks from EDA ──────────────────────────────────────────────────────
BENCHMARKS = {
    'min_training_bonus':  {'good': 8.0,  'great': 12.0, 'dir': '+', 'label': 'Min training bonus (weakest member)'},
    'ghost_count':         {'good': 1,    'great': 0,    'dir': '-', 'label': 'Ghost members (>14d inactive)'},
    'inactive_count':      {'good': 1,    'great': 0,    'dir': '-', 'label': 'Inactive members (>7d)'},
    'full_attendance':     {'good': 0.5,  'great': 1.0,  'dir': '+', 'label': 'Full attendance today'},
    'mean_training_count': {'good': 130,  'great': 150,  'dir': '+', 'label': 'Avg training sessions (28d)'},
    'bonus_cv':            {'good': 0.5,  'great': 0.2,  'dir': '-', 'label': 'Bonus spread (lower = more uniform)'},
    'mean_stars_top11':    {'good': 5.5,  'great': 6.5,  'dir': '+', 'label': 'Avg squad quality (stars)'},
    'training_efficiency': {'good': 5.0,  'great': 7.0,  'dir': '+', 'label': 'Training efficiency'},
    'sum_multiplier':      {'good': 14,   'great': 18,   'dir': '+', 'label': 'Total clan multiplier'},
}

def score_gap(value, bench):
    good, great, direction = bench['good'], bench['great'], bench['dir']
    if direction == '+':
        if value >= great: return 'great', None
        if value >= good:  return 'ok',    f'{value:.2f} → target {great:.2f}'
        return 'poor',                     f'critical: {value:.2f} vs minimum {good:.2f}'
    else:
        if value <= great: return 'great', None
        if value <= good:  return 'ok',    f'{value:.2f} → target {great:.2f}'
        return 'poor',                     f'critical: {value:.2f} vs threshold {good:.2f}'


def clan_report(clan_id, members_df, clan_agg):
    players = members_df[members_df['clan_id'] == clan_id].copy()
    if players.empty or clan_id not in clan_agg.index:
        return f'Clan {clan_id} not found.'
    feats = clan_agg.loc[clan_id]

    lines = [f'\n{"="*60}', f'  Advisory Report — {clan_id}', f'{"="*60}']

    # Ghost / inactive members
    ghosts   = players[players['days_since_last_active'] > 14].sort_values('days_since_last_active', ascending=False)
    inactive = players[(players['days_since_last_active'] > 7) & (players['days_since_last_active'] <= 14)]

    if not ghosts.empty:
        lines.append(f'\n[!!] GHOST MEMBERS — {len(ghosts)} player(s) missing >14 days:')
        for _, p in ghosts.iterrows():
            lines.append(f'     {p["user_id"]:20s}  absent={int(p["days_since_last_active"])}d  '
                         f'stars={p["avg_stars_top_11_players"]:.1f}  '
                         f'bonus={p["avg_training_bonus"]:.1f}  '
                         f'multiplier=x{int(p["clan_multiplier"])}')
        lines.append('     → Each ghost costs 6–18 points per tournament. Contact or replace ASAP.')

    if not inactive.empty:
        lines.append(f'\n[!]  AT-RISK — {len(inactive)} player(s) inactive 7–14 days:')
        for _, p in inactive.iterrows():
            lines.append(f'     {p["user_id"]:20s}  absent={int(p["days_since_last_active"])}d')
        lines.append('     → Contact before the tournament.')

    # Bonus floor
    low_bonus = players[players['avg_training_bonus'] < 5].sort_values('avg_training_bonus')
    if not low_bonus.empty:
        lines.append(f'\n[!]  LOW TRAINING BONUS — {len(low_bonus)} player(s) below 5:')
        for _, p in low_bonus.iterrows():
            lines.append(f'     {p["user_id"]:20s}  bonus={p["avg_training_bonus"]:.1f}')
        lines.append('     → Training bonus is the #1 win predictor. Clans with floor ≥8 win 59% vs 45%.')

    # Per-benchmark scoring
    issues, strengths = [], []
    for key, bench in BENCHMARKS.items():
        val = feats.get(key)
        if val is None: continue
        rating, gap = score_gap(float(val), bench)
        if rating == 'poor':  issues.append(('CRITICAL', bench['label'], gap))
        elif rating == 'ok':  issues.append(('OK',       bench['label'], gap))
        else:                 strengths.append(bench['label'])

    critical = [i for i in issues if i[0] == 'CRITICAL']
    ok_items = [i for i in issues if i[0] == 'OK']

    if critical:
        lines.append('\n[PRIORITY 1 — Fix These First]:')
        for _, label, gap in critical:
            lines.append(f'  • {label}: {gap}')

    if ok_items:
        lines.append('\n[PRIORITY 2 — Room to Improve]:')
        for _, label, gap in ok_items:
            lines.append(f'  • {label}: {gap}')

    if strengths:
        lines.append('\n[STRENGTHS — Keep These Up]:')
        for s in strengths:
            lines.append(f'  ✓ {s}')

    return '\n'.join(lines)


print('Advisory functions loaded.')

In [ ]:
# ── Demo: 3 different clan profiles ──────────────────────────────────────────

# Clan 1: worst performers (most ghost members)
ghost_heavy = members_train.groupby('clan_id').apply(
    lambda d: (d['days_since_last_active'] > 14).sum()
).sort_values(ascending=False)
clan_worst = ghost_heavy[ghost_heavy >= 3].index[0]

# Clan 2: best performers (low ghost count + high bonus)
strong_clans = clan_agg_train[
    (clan_agg_train['ghost_count'] == 0) &
    (clan_agg_train['min_training_bonus'] > 12) &
    (clan_agg_train['full_attendance'] == 1)
].index
clan_best = strong_clans[0] if len(strong_clans) > 0 else members_train['clan_id'].iloc[0]

# Clan 3: typical clan (close to dataset mean)
mean_ghost = clan_agg_train['ghost_count'].mean()
mean_bonus = clan_agg_train['mean_training_bonus'].mean()
typical = clan_agg_train[
    (clan_agg_train['ghost_count'].between(mean_ghost-0.3, mean_ghost+0.3)) &
    (clan_agg_train['mean_training_bonus'].between(mean_bonus-0.5, mean_bonus+0.5))
].index
clan_typical = typical[0] if len(typical) > 0 else members_train['clan_id'].iloc[100]

for label, cid in [('STRUGGLING CLAN', clan_worst),
                   ('STRONG CLAN',     clan_best),
                   ('TYPICAL CLAN',    clan_typical)]:
    print(f'\n>>> {label}: {cid}')
    print(clan_report(cid, members_train, clan_agg_train))

In [ ]:
# ── Claude API Chatbot (requires ANTHROPIC_API_KEY) ───────────────────────────
# Install: !pip install anthropic -q
# Set key: import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

SYSTEM_PROMPT = """You are a Top Eleven clan tournament advisor powered by data from 24,000+ real matches.

Game mechanics:
- 6 members per clan, each plays 2 matches (12 total)
- Points: Win=3, Draw=1, Loss=0 — MULTIPLIED by that player's clan_multiplier
- One ghost member (absent >14 days) contributes 0 points regardless of quality

Data-proven priorities (ordered by impact):
1. Min training bonus — strongest single predictor. 0 floor → 45% WR, 12+ floor → 59% WR
2. Full attendance — all 6 logged in today adds +9pp win rate
3. Ghost members — each one reduces win rate ~5%
4. Bonus uniformity — uneven teams underperform vs even teams
5. Training efficiency — winners train 19% more per active day
6. Squad quality — matters but less than engagement
7. Multiplier carry (one strong player) — does NOT work, confirmed by data

Be specific, mention player IDs when relevant, quantify impact where possible.
Keep answers concise and actionable."""

TOOLS = [
    {
        'name': 'get_clan_report',
        'description': 'Get full advisory report for a clan by ID.',
        'input_schema': {
            'type': 'object',
            'properties': {'clan_id': {'type': 'string'}},
            'required': ['clan_id']
        }
    },
    {
        'name': 'compare_clans',
        'description': 'Compare two clans head-to-head on key metrics.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'clan_1_id': {'type': 'string'},
                'clan_2_id': {'type': 'string'}
            },
            'required': ['clan_1_id', 'clan_2_id']
        }
    }
]

def execute_tool(name, args):
    if name == 'get_clan_report':
        return clan_report(args['clan_id'], members_train, clan_agg_train)
    elif name == 'compare_clans':
        c1_id, c2_id = args['clan_1_id'], args['clan_2_id']
        if c1_id not in clan_agg_train.index or c2_id not in clan_agg_train.index:
            return 'One or both clans not found.'
        keys = ['min_training_bonus', 'ghost_count', 'full_attendance',
                'mean_training_count', 'mean_stars_top11', 'sum_multiplier',
                'bonus_cv', 'inactive_count']
        rows = []
        for k in keys:
            v1, v2 = float(clan_agg_train.loc[c1_id, k]), float(clan_agg_train.loc[c2_id, k])
            rows.append(f'{k:30s}: clan_1={v1:.2f}  clan_2={v2:.2f}')
        return '\n'.join(rows)
    return 'Unknown tool.'


def ask_advisor(question, history=None):
    try:
        import anthropic
        api_key = os.environ.get('ANTHROPIC_API_KEY', '')
        if not api_key:
            return '[No API key — showing rule-based answer]\n' + _rule_based_answer(question)
        client = anthropic.Anthropic(api_key=api_key)
        messages = list(history or [])
        messages.append({'role': 'user', 'content': question})
        while True:
            response = client.messages.create(
                model='claude-sonnet-4-6', max_tokens=1024,
                system=SYSTEM_PROMPT, tools=TOOLS, messages=messages
            )
            tool_uses = [b for b in response.content if b.type == 'tool_use']
            if not tool_uses:
                text = next((b.text for b in response.content if b.type == 'text'), '')
                messages.append({'role': 'assistant', 'content': response.content})
                return text, messages
            messages.append({'role': 'assistant', 'content': response.content})
            results = [{'type': 'tool_result', 'tool_use_id': t.id,
                        'content': execute_tool(t.name, t.input)} for t in tool_uses]
            messages.append({'role': 'user', 'content': results})
    except ImportError:
        return '[anthropic not installed]\n' + _rule_based_answer(question), []


def _rule_based_answer(question):
    q = question.lower()
    for token in q.split():
        if token.startswith('clan_'):
            return clan_report(token, members_train, clan_agg_train)
    return 'Please mention a clan ID in your question (e.g. clan_5029188).'


print('Chatbot ready.')
print('Usage: answer, history = ask_advisor("your question here")')
print('       answer, history = ask_advisor("follow-up", history=history)')

In [ ]:
# ── Example conversation ──────────────────────────────────────────────────────
print('=== CHATBOT TEST CONVERSATION ===')
print(f'Using clan: {clan_worst} (struggling clan from demo above)\n')

q1 = f'What are the top 3 things {clan_worst} should do before their next tournament?'
print(f'Q: {q1}')
result = ask_advisor(q1)
if isinstance(result, tuple):
    answer1, history = result
else:
    answer1, history = result, []
print(f'A: {answer1}')

print('\n' + '-'*60 + '\n')

q2 = 'Which specific member should they focus on first, and why?'
print(f'Q: {q2}')
result2 = ask_advisor(q2, history=history)
if isinstance(result2, tuple):
    answer2, _ = result2
else:
    answer2 = result2
print(f'A: {answer2}')

print('\n' + '-'*60 + '\n')

q3 = f'Compare {clan_worst} vs {clan_best} — who is likely to win and by how much?'
print(f'Q: {q3}')
result3 = ask_advisor(q3)
if isinstance(result3, tuple):
    answer3, _ = result3
else:
    answer3 = result3
print(f'A: {answer3}')

---
## Summary

### Model Performance

| Version | Model | CV Accuracy | Std |
|---|---|---|---|
| v1 | XGB + LGB ensemble | 0.5851 | ±0.0136 |
| **v2 (this notebook)** | **LGB + Ordinal blend** | **0.5836** | **±0.0074** |

Mean accuracy is essentially the same; variance is halved. The v2 model is significantly more reliable on hard matches (fold 2: 0.5624 → 0.5731).

### What drives clan tournament outcomes

1. **Training bonus floor** — the single strongest predictor. Fix the weakest member first.
2. **Attendance** — ghost members are catastrophic; full attendance is a +9pp boost.
3. **Team consistency** — uniform bonus and activity levels outperform uneven teams.
4. **Quality matters, but less than you'd expect** — an active mediocre team beats an inactive star-studded one.

### Advisory System

The chatbot translates model insights into specific, player-level recommendations. It can answer free-text questions via the Claude API and falls back to a rule-based system without any API key. See `chatbot.py` for the full interactive command-line version.